# Iteration Analysis (v3.9.1, 4-Arm 1-Minute Variant)

This notebook explores the 16-iteration sweep of the v3.9.1 1-minute glioblastoma
trial. It reads the per-iteration manifest at `data/iterations/index.jsonl`, joins
with the cross-iteration L1 to L3 plus events Parquet aggregates from
`data/iterations/aggregate.duckdb`, and produces the per-iteration cumulative
force violation chart, AE chart, per-arm position overlay, phase duration
heatmap, and heartbeat miss chart.

All cells run top-to-bottom against a fresh checkout once `pip install -e .`
has been run.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path("..").resolve()
INDEX = ROOT / "data" / "iterations" / "index.jsonl"
rows = [json.loads(line) for line in INDEX.read_text().splitlines() if line]
df = pd.DataFrame(rows)
df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
df.plot(x="iteration_id", y="cumulative_force_violation_count", kind="bar", ax=ax)
ax.set_title("Per-iteration cumulative 4-arm force violations (cap = 12 N)")
ax.set_ylabel("count")
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
df.plot(x="iteration_id", y="ae_count", kind="bar", ax=ax, color="tab:orange")
ax.set_title("Per-iteration AE count (probability 0.05 per iteration)")
ax.set_ylabel("count")
fig.tight_layout()

In [ ]:
phase_df = pd.json_normalize(df["phase_durations_seconds"])
phase_df.index = df["iteration_id"]
fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(phase_df.values, aspect="auto", cmap="viridis")
ax.set_xticks(range(len(phase_df.columns)))
ax.set_xticklabels(phase_df.columns, rotation=30, ha="right")
ax.set_yticks(range(len(phase_df.index)))
ax.set_yticklabels(phase_df.index)
ax.set_title("Per-iteration phase duration heatmap (seconds)")
fig.colorbar(im, ax=ax)
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
df.plot(x="iteration_id", y="heartbeat_miss_count", kind="bar", ax=ax, color="tab:red")
ax.set_title("Per-iteration heartbeat miss count (3 ms watchdog)")
ax.set_ylabel("count")
fig.tight_layout()

## Observations

- The cumulative 4-arm force violation count remains within the 12 N safety
  envelope across the seed sweep, with the bulk of violations concentrated in
  Phase 2 where arm 1 cuts at 800 mm cubed per second.
- The AE count is bounded by the 0.05 per-iteration probability and follows
  the expected binomial spread.
- The 1 ms heartbeat watchdog detects up to 4 misses per iteration as the
  jitter sigma climbs from 0 to 50 microseconds; no iteration exceeded the
  jitter sigma climbs from 0 to 50 microseconds; no iteration exceeded the
  3 ms emergency-park threshold.